# Testing Finviz Screener Agent

This notebook tests the `finviz_screener_agent` functionality in isolation to debug screening issues.

## Setup
- Load environment variables
- Import required libraries
- Test finviz screener functionality


In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")
print(f"Project root: {project_root}")
print(f"Python path includes project root: {str(project_root) in sys.path}")


## Test Screener Agent Function

Now let's test our actual screener agent function.


In [5]:
# Test the actual screener agent
try:
    from src.agents.finviz_screener_agent import finviz_screener_node
    from src.state import SMAState
    print("✓ Screener agent imported successfully")
    
    # Create a test state
    test_state: SMAState = {
        "structured_constraints": {
            "excluded_sectors": []  # No exclusions for testing
        }
    }
    
    print("\nTesting screener agent with empty constraints...")
    result = finviz_screener_node(test_state)
    
    print("Result keys:", list(result.keys()))
    print(f"Universe size: {len(result.get('universe', []))}")
    
    if result.get('universe'):
        print("Sample tickers:", result['universe'][:10])
    
    if result.get('feedback'):
        print(f"Feedback: {result['feedback']}")
        
    if result.get('screener_criteria'):
        print(f"Screener criteria: {result['screener_criteria']}")
    
except Exception as e:
    print(f"✗ Error testing screener agent: {e}")
    import traceback
    traceback.print_exc()


✓ Screener agent imported successfully

Testing screener agent with empty constraints...
🔍 Screening stocks with finviz...
Criteria: Market Cap > Large, EPS > 0
Error during stock screening: list index out of range
Result keys: ['universe', 'feedback']
Universe size: 0
Feedback: Error during stock screening: list index out of range


## Test with Sector Filtering

Let's test the screener agent with sector exclusions.


In [4]:
# Test with sector filtering
try:
    from src.agents.finviz_screener_agent import finviz_screener_node
    from src.state import SMAState
    print("✓ Screener agent imported successfully")
    
    test_state_with_exclusions: SMAState = {
        "structured_constraints": {
            "excluded_sectors": []  # Exclude some sectors
        }
    }
    
    print("Testing screener agent with sector exclusions...")
    result = finviz_screener_node(test_state_with_exclusions)
    
    print("Result keys:", list(result.keys()))
    print(f"Universe size: {len(result.get('universe', []))}")
    
    if result.get('universe'):
        print("Sample tickers:", result['universe'][:10])
    
    if result.get('feedback'):
        print(f"Feedback: {result['feedback']}")
        
    if result.get('screener_criteria'):
        print(f"Screener criteria: {result['screener_criteria']}")
    
except Exception as e:
    print(f"Error testing with sector filtering: {e}")
    import traceback
    traceback.print_exc()


✓ Screener agent imported successfully
Testing screener agent with sector exclusions...
🔍 Screening stocks with finviz...
Criteria: Market Cap > Large, EPS > 0
Error during stock screening: list index out of range
Result keys: ['universe', 'feedback']
Universe size: 0
Feedback: Error during stock screening: list index out of range


## Debug YFinance Data Structure

Let's examine the data structure returned by yfinance.


In [ ]:
# Debug the yfinance data structure
try:
    print("Examining yfinance stock info structure...")
    
    stock = yf.Ticker("AAPL")
    info = stock.info
    
    print(f"Number of info fields: {len(info)}")
    print("Key fields:")
    
    important_fields = ['longName', 'sector', 'industry', 'marketCap', 'trailingEps', 'forwardEps']
    for field in important_fields:
        value = info.get(field, 'Not available')
        print(f"  {field}: {value}")
        
except Exception as e:
    print(f"Error examining data structure: {e}")
    import traceback
    traceback.print_exc()


## Test Pricing Agent Integration

Let's also test that the pricing agent can work with the screened stocks.


In [ ]:
# Test pricing agent with screened results
try:
    from src.agents.pricing_agent import pricing_agent_node
    
    # First get some stocks from screener
    screener_result = finviz_screener_node({
        "structured_constraints": {"excluded_sectors": []}
    })
    
    universe = screener_result.get('universe', [])[:5]  # Just test with first 5
    
    if universe:
        print(f"Testing pricing agent with {len(universe)} stocks: {universe}")
        
        pricing_state = {"universe": universe}
        pricing_result = pricing_agent_node(pricing_state)
        
        market_data = pricing_result.get('market_data')
        if market_data is not None and not market_data.empty:
            print("✓ Pricing agent successful!")
            print(f"  Data shape: {market_data.shape}")
            print(f"  Date range: {market_data.index.min()} to {market_data.index.max()}")
            print(f"  Columns: {list(market_data.columns)}")
        else:
            print("✗ Pricing agent returned empty data")
            print(f"Feedback: {pricing_result.get('feedback')}")
    else:
        print("No stocks from screener to test pricing")
    
except Exception as e:
    print(f"Error testing pricing integration: {e}")
    import traceback
    traceback.print_exc()


## Summary and Troubleshooting

This notebook tests the stock screening system. Key changes made:

1. **Finviz Issue**: The finviz library is currently broken due to website scraping failures
2. **YFinance Fallback**: Switched to using yfinance for stock screening with a predefined list of large-cap stocks
3. **Reliable Screening**: YFinance provides more reliable data access

**Current Screening Criteria:**
- Market Cap > $10B
- EPS > 0
- Sector filtering based on mandate constraints

**Troubleshooting:**
- If no stocks are found, check yfinance data availability
- Verify network connectivity for yfinance API calls
- Check that excluded sectors aren't filtering out all stocks


In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")
print(f"Project root: {project_root}")
print(f"Python path includes project root: {str(project_root) in sys.path}")


## Test Finviz Library Basics

First, let's test the basic finviz functionality to ensure it's working.


In [ ]:
# Test basic finviz imports and functionality
try:
    from finviz.screener import Screener
    print("✓ Finviz Screener imported successfully")
    
    # Test basic screener without filters
    print("\nTesting basic screener...")
    basic_screener = Screener(rows=5)  # Just get 5 rows
    print(f"✓ Basic screener created with {len(basic_screener.data)} results")
    
    if basic_screener.data:
        print("Sample data:")
        for i, stock in enumerate(basic_screener.data[:3]):
            print(f"  {i+1}. {stock.get('Ticker', 'N/A')} - {stock.get('Company', 'N/A')} - {stock.get('Sector', 'N/A')}")
    
except Exception as e:
    print(f"✗ Error with finviz: {e}")
    import traceback
    traceback.print_exc()


## Test Different Filter Combinations

Let's test various filter combinations to see what works.


In [ ]:
# Test different filter combinations
test_filters = [
    ('cap_large_over', 'Large cap only'),
    ('eps_positive', 'Positive EPS only'),
    ('cap_large_over,eps_positive', 'Large cap + positive EPS'),
    ('cap_mid_over', 'Mid cap and above'),
    ('cap_small_over', 'Small cap and above'),
]

for filter_str, description in test_filters:
    try:
        print(f"\n{'='*60}")
        print(f"Testing: {description}")
        print(f"Filters: {filter_str}")
        print('='*60)
        
        filters = filter_str.split(',') if ',' in filter_str else [filter_str]
        screener = Screener(filters=filters, rows=10)
        
        print(f"Results found: {len(screener.data)}")
        
        if screener.data:
            print("Sample results:")
            for i, stock in enumerate(screener.data[:5]):
                ticker = stock.get('Ticker', 'N/A')
                company = stock.get('Company', 'N/A')
                sector = stock.get('Sector', 'N/A')
                market_cap = stock.get('Market Cap', 'N/A')
                eps = stock.get('EPS', 'N/A')
                print(f"  {i+1}. {ticker} ({company}) - Sector: {sector} - MCap: {market_cap} - EPS: {eps}")
        else:
            print("No results found!")
            
    except Exception as e:
        print(f"Error testing {description}: {e}")


## Test Screener Agent Function

Now let's test our actual screener agent function.


In [ ]:
# Test the actual screener agent
try:
    from src.agents.finviz_screener_agent import finviz_screener_node
    from src.state import SMAState
    print("✓ Screener agent imported successfully")
    
    # Create a test state
    test_state: SMAState = {
        "structured_constraints": {
            "excluded_sectors": []  # No exclusions for testing
        }
    }
    
    print("\nTesting screener agent with empty constraints...")
    result = finviz_screener_node(test_state)
    
    print("Result keys:", list(result.keys()))
    print(f"Universe size: {len(result.get('universe', []))}")
    
    if result.get('universe'):
        print("Sample tickers:", result['universe'][:10])
    
    if result.get('feedback'):
        print(f"Feedback: {result['feedback']}")
        
    if result.get('screener_criteria'):
        print(f"Screener criteria: {result['screener_criteria']}")
    
except Exception as e:
    print(f"✗ Error testing screener agent: {e}")
    import traceback
    traceback.print_exc()


## Test with Sector Filtering

Let's test the screener agent with sector exclusions.


In [ ]:
# Test with sector filtering
try:
    test_state_with_exclusions: SMAState = {
        "structured_constraints": {
            "excluded_sectors": ["Technology", "Healthcare"]  # Exclude some sectors
        }
    }
    
    print("Testing screener agent with sector exclusions...")
    result = finviz_screener_node(test_state_with_exclusions)
    
    print("Result keys:", list(result.keys()))
    print(f"Universe size: {len(result.get('universe', []))}")
    
    if result.get('universe'):
        print("Sample tickers:", result['universe'][:10])
    
    if result.get('feedback'):
        print(f"Feedback: {result['feedback']}")
        
    if result.get('screener_criteria'):
        print(f"Screener criteria: {result['screener_criteria']}")
    
except Exception as e:
    print(f"Error testing with sector filtering: {e}")
    import traceback
    traceback.print_exc()


## Debug Finviz Screener Data Structure

Let's examine the raw data structure returned by finviz to understand the format.


In [ ]:
# Debug the data structure
try:
    print("Examining finviz screener data structure...")
    
    # Get a small sample
    debug_screener = Screener(filters=['cap_large_over'], rows=3)
    
    if debug_screener.data:
        print(f"Number of results: {len(debug_screener.data)}")
        print(f"Type of data: {type(debug_screener.data)}")
        
        # Examine first result
        first_stock = debug_screener.data[0]
        print(f"\nFirst stock keys: {list(first_stock.keys())}")
        print("First stock data:")
        for key, value in first_stock.items():
            print(f"  {key}: {value}")
    else:
        print("No data to examine!")
        
except Exception as e:
    print(f"Error examining data structure: {e}")
    import traceback
    traceback.print_exc()


## Test Individual Stock Data

Let's also test getting individual stock data from finviz.


In [ ]:
# Test individual stock data
try:
    import finviz
    
    print("Testing individual stock data retrieval...")
    
    # Test with a well-known stock
    test_ticker = "AAPL"
    stock_data = finviz.get_stock(test_ticker)
    
    print(f"Stock data for {test_ticker}:")
    print(f"  Company: {stock_data.get('Company', 'N/A')}")
    print(f"  Sector: {stock_data.get('Sector', 'N/A')}")
    print(f"  Industry: {stock_data.get('Industry', 'N/A')}")
    print(f"  Market Cap: {stock_data.get('Market Cap', 'N/A')}")
    print(f"  EPS: {stock_data.get('EPS', 'N/A')}")
    
except Exception as e:
    print(f"Error testing individual stock data: {e}")
    import traceback
    traceback.print_exc()


## Summary and Troubleshooting

This notebook helps debug finviz screener issues. Common problems:

1. **No results**: Filters might be too restrictive
2. **Import errors**: finviz package might not be installed correctly
3. **Data format issues**: finviz API might have changed

If you're still getting no results, try:
- Using fewer/more permissive filters
- Checking if finviz service is available
- Verifying network connectivity


In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")
print(f"Project root: {project_root}")
print(f"Python path includes project root: {str(project_root) in sys.path}")


## Test Finviz Library Basics

First, let's test the basic finviz functionality to ensure it's working.


In [ ]:
# Test basic finviz imports and functionality
try:
    from finviz.screener import Screener
    print("✓ Finviz Screener imported successfully")
    
    # Test basic screener without filters
    print("\nTesting basic screener...")
    basic_screener = Screener(rows=5)  # Just get 5 rows
    print(f"✓ Basic screener created with {len(basic_screener.data)} results")
    
    if basic_screener.data:
        print("Sample data:")
        for i, stock in enumerate(basic_screener.data[:3]):
            print(f"  {i+1}. {stock.get('Ticker', 'N/A')} - {stock.get('Company', 'N/A')} - {stock.get('Sector', 'N/A')}")
    
except Exception as e:
    print(f"✗ Error with finviz: {e}")
    import traceback
    traceback.print_exc()


## Test Different Filter Combinations

Let's test various filter combinations to see what works.


In [ ]:
# Test different filter combinations
test_filters = [
    ('cap_large_over', 'Large cap only'),
    ('eps_positive', 'Positive EPS only'),
    ('cap_large_over,eps_positive', 'Large cap + positive EPS'),
    ('cap_mid_over', 'Mid cap and above'),
    ('cap_small_over', 'Small cap and above'),
]

for filter_str, description in test_filters:
    try:
        print(f"\n{'='*60}")
        print(f"Testing: {description}")
        print(f"Filters: {filter_str}")
        print('='*60)
        
        filters = filter_str.split(',') if ',' in filter_str else [filter_str]
        screener = Screener(filters=filters, rows=10)
        
        print(f"Results found: {len(screener.data)}")
        
        if screener.data:
            print("Sample results:")
            for i, stock in enumerate(screener.data[:5]):
                ticker = stock.get('Ticker', 'N/A')
                company = stock.get('Company', 'N/A')
                sector = stock.get('Sector', 'N/A')
                market_cap = stock.get('Market Cap', 'N/A')
                eps = stock.get('EPS', 'N/A')
                print(f"  {i+1}. {ticker} ({company}) - Sector: {sector} - MCap: {market_cap} - EPS: {eps}")
        else:
            print("No results found!")
            
    except Exception as e:
        print(f"Error testing {description}: {e}")


## Test Screener Agent Function

Now let's test our actual screener agent function.


In [ ]:
# Test the actual screener agent
try:
    from src.agents.finviz_screener_agent import finviz_screener_node
    from src.state import SMAState
    print("✓ Screener agent imported successfully")
    
    # Create a test state
    test_state: SMAState = {
        "structured_constraints": {
            "excluded_sectors": []  # No exclusions for testing
        }
    }
    
    print("\nTesting screener agent with empty constraints...")
    result = finviz_screener_node(test_state)
    
    print("Result keys:", list(result.keys()))
    print(f"Universe size: {len(result.get('universe', []))}")
    
    if result.get('universe'):
        print("Sample tickers:", result['universe'][:10])
    
    if result.get('feedback'):
        print(f"Feedback: {result['feedback']}")
        
    if result.get('screener_criteria'):
        print(f"Screener criteria: {result['screener_criteria']}")
    
except Exception as e:
    print(f"✗ Error testing screener agent: {e}")
    import traceback
    traceback.print_exc()


## Test with Sector Filtering

Let's test the screener agent with sector exclusions.


In [ ]:
# Test with sector filtering
try:
    test_state_with_exclusions: SMAState = {
        "structured_constraints": {
            "excluded_sectors": ["Technology", "Healthcare"]  # Exclude some sectors
        }
    }
    
    print("Testing screener agent with sector exclusions...")
    result = finviz_screener_node(test_state_with_exclusions)
    
    print("Result keys:", list(result.keys()))
    print(f"Universe size: {len(result.get('universe', []))}")
    
    if result.get('universe'):
        print("Sample tickers:", result['universe'][:10])
    
    if result.get('feedback'):
        print(f"Feedback: {result['feedback']}")
        
    if result.get('screener_criteria'):
        print(f"Screener criteria: {result['screener_criteria']}")
    
except Exception as e:
    print(f"Error testing with sector filtering: {e}")
    import traceback
    traceback.print_exc()


## Debug Finviz Screener Data Structure

Let's examine the raw data structure returned by finviz to understand the format.


In [ ]:
# Debug the data structure
try:
    print("Examining finviz screener data structure...")
    
    # Get a small sample
    debug_screener = Screener(filters=['cap_large_over'], rows=3)
    
    if debug_screener.data:
        print(f"Number of results: {len(debug_screener.data)}")
        print(f"Type of data: {type(debug_screener.data)}")
        
        # Examine first result
        first_stock = debug_screener.data[0]
        print(f"\nFirst stock keys: {list(first_stock.keys())}")
        print("First stock data:")
        for key, value in first_stock.items():
            print(f"  {key}: {value}")
    else:
        print("No data to examine!")
        
except Exception as e:
    print(f"Error examining data structure: {e}")
    import traceback
    traceback.print_exc()


## Test Individual Stock Data

Let's also test getting individual stock data from finviz.


In [ ]:
# Test individual stock data
try:
    import finviz
    
    print("Testing individual stock data retrieval...")
    
    # Test with a well-known stock
    test_ticker = "AAPL"
    stock_data = finviz.get_stock(test_ticker)
    
    print(f"Stock data for {test_ticker}:")
    print(f"  Company: {stock_data.get('Company', 'N/A')}")
    print(f"  Sector: {stock_data.get('Sector', 'N/A')}")
    print(f"  Industry: {stock_data.get('Industry', 'N/A')}")
    print(f"  Market Cap: {stock_data.get('Market Cap', 'N/A')}")
    print(f"  EPS: {stock_data.get('EPS', 'N/A')}")
    
except Exception as e:
    print(f"Error testing individual stock data: {e}")
    import traceback
    traceback.print_exc()


## Summary and Troubleshooting

This notebook helps debug finviz screener issues. Common problems:

1. **No results**: Filters might be too restrictive
2. **Import errors**: finviz package might not be installed correctly
3. **Data format issues**: finviz API might have changed

If you're still getting no results, try:
- Using fewer/more permissive filters
- Checking if finviz service is available
- Verifying network connectivity


## Test Finviz Library Basics

First, let's test the basic finviz functionality to ensure it's working.


In [ ]:
# Test basic finviz imports and functionality
try:
    from finviz.screener import Screener
    print("✓ Finviz Screener imported successfully")
    
    # Test basic screener without filters
    print("\nTesting basic screener...")
    basic_screener = Screener(rows=5)  # Just get 5 rows
    print(f"✓ Basic screener created with {len(basic_screener.data)} results")
    
    if basic_screener.data:
        print("Sample data:")
        for i, stock in enumerate(basic_screener.data[:3]):
            print(f"  {i+1}. {stock.get('Ticker', 'N/A')} - {stock.get('Company', 'N/A')} - {stock.get('Sector', 'N/A')}")
    
except Exception as e:
    print(f"✗ Error with finviz: {e}")
    import traceback
    traceback.print_exc()


## Test Different Filter Combinations

Let's test various filter combinations to see what works.


In [ ]:
# Test different filter combinations
test_filters = [
    ('cap_large_over', 'Large cap only'),
    ('eps_positive', 'Positive EPS only'),
    ('cap_large_over,eps_positive', 'Large cap + positive EPS'),
    ('cap_mid_over', 'Mid cap and above'),
    ('cap_small_over', 'Small cap and above'),
]

for filter_str, description in test_filters:
    try:
        print(f"\n{'='*60}")
        print(f"Testing: {description}")
        print(f"Filters: {filter_str}")
        print('='*60)
        
        filters = filter_str.split(',') if ',' in filter_str else [filter_str]
        screener = Screener(filters=filters, rows=10)
        
        print(f"Results found: {len(screener.data)}")
        
        if screener.data:
            print("Sample results:")
            for i, stock in enumerate(screener.data[:5]):
                ticker = stock.get('Ticker', 'N/A')
                company = stock.get('Company', 'N/A')
                sector = stock.get('Sector', 'N/A')
                market_cap = stock.get('Market Cap', 'N/A')
                eps = stock.get('EPS', 'N/A')
                print(f"  {i+1}. {ticker} ({company}) - Sector: {sector} - MCap: {market_cap} - EPS: {eps}")
        else:
            print("No results found!")
            
    except Exception as e:
        print(f"Error testing {description}: {e}")


## Test Screener Agent Function

Now let's test our actual screener agent function.


In [ ]:
# Test the actual screener agent
try:
    from src.agents.finviz_screener_agent import finviz_screener_node
    from src.state import SMAState
    print("✓ Screener agent imported successfully")
    
    # Create a test state
    test_state: SMAState = {
        "structured_constraints": {
            "excluded_sectors": []  # No exclusions for testing
        }
    }
    
    print("\nTesting screener agent with empty constraints...")
    result = finviz_screener_node(test_state)
    
    print("Result keys:", list(result.keys()))
    print(f"Universe size: {len(result.get('universe', []))}")
    
    if result.get('universe'):
        print("Sample tickers:", result['universe'][:10])
    
    if result.get('feedback'):
        print(f"Feedback: {result['feedback']}")
        
    if result.get('screener_criteria'):
        print(f"Screener criteria: {result['screener_criteria']}")
    
except Exception as e:
    print(f"✗ Error testing screener agent: {e}")
    import traceback
    traceback.print_exc()


## Test with Sector Filtering

Let's test the screener agent with sector exclusions.


In [ ]:
# Test with sector filtering
try:
    test_state_with_exclusions: SMAState = {
        "structured_constraints": {
            "excluded_sectors": ["Technology", "Healthcare"]  # Exclude some sectors
        }
    }
    
    print("Testing screener agent with sector exclusions...")
    result = finviz_screener_node(test_state_with_exclusions)
    
    print("Result keys:", list(result.keys()))
    print(f"Universe size: {len(result.get('universe', []))}")
    
    if result.get('universe'):
        print("Sample tickers:", result['universe'][:10])
    
    if result.get('feedback'):
        print(f"Feedback: {result['feedback']}")
        
    if result.get('screener_criteria'):
        print(f"Screener criteria: {result['screener_criteria']}")
    
except Exception as e:
    print(f"Error testing with sector filtering: {e}")
    import traceback
    traceback.print_exc()


## Debug Finviz Screener Data Structure

Let's examine the raw data structure returned by finviz to understand the format.


In [ ]:
# Debug the data structure
try:
    print("Examining finviz screener data structure...")
    
    # Get a small sample
    debug_screener = Screener(filters=['cap_large_over'], rows=3)
    
    if debug_screener.data:
        print(f"Number of results: {len(debug_screener.data)}")
        print(f"Type of data: {type(debug_screener.data)}")
        
        # Examine first result
        first_stock = debug_screener.data[0]
        print(f"\nFirst stock keys: {list(first_stock.keys())}")
        print("First stock data:")
        for key, value in first_stock.items():
            print(f"  {key}: {value}")
    else:
        print("No data to examine!")
        
except Exception as e:
    print(f"Error examining data structure: {e}")
    import traceback
    traceback.print_exc()


## Test Individual Stock Data

Let's also test getting individual stock data from finviz.


In [ ]:
# Test individual stock data
try:
    import finviz
    
    print("Testing individual stock data retrieval...")
    
    # Test with a well-known stock
    test_ticker = "AAPL"
    stock_data = finviz.get_stock(test_ticker)
    
    print(f"Stock data for {test_ticker}:")
    print(f"  Company: {stock_data.get('Company', 'N/A')}")
    print(f"  Sector: {stock_data.get('Sector', 'N/A')}")
    print(f"  Industry: {stock_data.get('Industry', 'N/A')}")
    print(f"  Market Cap: {stock_data.get('Market Cap', 'N/A')}")
    print(f"  EPS: {stock_data.get('EPS', 'N/A')}")
    
except Exception as e:
    print(f"Error testing individual stock data: {e}")
    import traceback
    traceback.print_exc()


## Summary and Troubleshooting

This notebook helps debug finviz screener issues. Common problems:

1. **No results**: Filters might be too restrictive
2. **Import errors**: finviz package might not be installed correctly
3. **Data format issues**: finviz API might have changed

If you're still getting no results, try:
- Using fewer/more permissive filters
- Checking if finviz service is available
- Verifying network connectivity
